# Elastic constants with `pyiron_workflow_atomistics`

This notebook computes the full elastic stiffness tensor and every derived constant in the
[Materials Project elasticity methodology](https://docs.materialsproject.org/methodology/materials-methodology/elasticity)
using `pyiron_workflow_atomistics.physics.elastic`, driven by **any ASE calculator**.

**Method (same as Materials Project, via `pymatgen.analysis.elasticity`):**
1. Relax the structure (cell + ions).
2. Apply the MP-standard deformation set (normal strains ±0.5%/±1%, shear ±3%/±6% → 24 cells).
3. Relax ions at **fixed cell** for each deformation and read the stress.
4. Fit the 6×6 stiffness tensor `C_ij`; derive K, G (Voigt/Reuss/Hill), Young's E, Poisson ν, universal anisotropy A_U, and a Born mechanical-stability check.


## 1. Fast example with the EMT calculator (Cu)
EMT is cheap and ships with ASE, so this cell runs end-to-end in ~20 s.

In [1]:
from ase.build import bulk
from ase.calculators.emt import EMT
from pyiron_workflow_atomistics.engine import ASEEngine, CalcInputStatic
from pyiron_workflow_atomistics.physics.elastic import calculate_elastic_constants

structure = bulk('Cu', 'fcc', a=3.615, cubic=True)
engine = ASEEngine(EngineInput=CalcInputStatic(), calculator=EMT(),
                   working_directory='elastic_emt_cu')
wf = calculate_elastic_constants(structure=structure, engine=engine, relax_initial=True)
out = wf.run()
d = out['elastic_constants']
print('mechanically stable:', d['mechanically_stable'])

      Step     Time          Energy          fmax
BFGS:    0 20:10:05       -0.019744        0.795278


      Step     Time          Energy          fmax
BFGS:    0 20:10:12       -0.025343        0.000000
      Step     Time          Energy          fmax
BFGS:    0 20:10:12       -0.023140        0.000000
      Step     Time          Energy          fmax
BFGS:    0 20:10:12       -0.015202        0.000000
      Step     Time          Energy          fmax
BFGS:    0 20:10:12       -0.009556        0.000000
      Step     Time          Energy          fmax
BFGS:    0 20:10:12       -0.025343        0.000000
      Step     Time          Energy          fmax
BFGS:    0 20:10:12       -0.023140        0.000000
      Step     Time          Energy          fmax
BFGS:    0 20:10:12       -0.015202        0.000000


      Step     Time          Energy          fmax
BFGS:    0 20:10:12       -0.009556        0.000000
      Step     Time          Energy          fmax
BFGS:    0 20:10:12       -0.025343        0.000000
      Step     Time          Energy          fmax
BFGS:    0 20:10:12       -0.023140        0.000000
      Step     Time          Energy          fmax
BFGS:    0 20:10:12       -0.015202        0.000000
      Step     Time          Energy          fmax
BFGS:    0 20:10:12       -0.009556        0.000000


      Step     Time          Energy          fmax
BFGS:    0 20:10:13        0.164385        0.000000
      Step     Time          Energy          fmax
BFGS:    0 20:10:13        0.023179        0.000000
      Step     Time          Energy          fmax
BFGS:    0 20:10:13        0.023179        0.000000
      Step     Time          Energy          fmax
BFGS:    0 20:10:13        0.164385        0.000000
      Step     Time          Energy          fmax
BFGS:    0 20:10:13        0.164385        0.000000


      Step     Time          Energy          fmax
BFGS:    0 20:10:13        0.023179        0.000000
      Step     Time          Energy          fmax
BFGS:    0 20:10:13        0.023179        0.000000
      Step     Time          Energy          fmax
BFGS:    0 20:10:13        0.164385        0.000000
      Step     Time          Energy          fmax
BFGS:    0 20:10:13        0.164385        0.000000
      Step     Time          Energy          fmax
BFGS:    0 20:10:13        0.023179        0.000000
      Step     Time          Energy          fmax
BFGS:    0 20:10:13        0.023179        0.000000
      Step     Time          Energy          fmax
BFGS:    0 20:10:13        0.164385        0.000000


mechanically stable: True


### The full 6×6 stiffness tensor (GPa, IEEE orientation)

In [2]:
import numpy as np, pandas as pd
C = np.array(d['elastic_tensor_ieee'])
pd.DataFrame(np.round(C, 1),
             index=[f'{i+1}' for i in range(6)],
             columns=[f'{j+1}' for j in range(6)])

,1,2,3,4,5,6
1,158.3,105.4,105.4,0.0,0.0,0.0
2,105.4,158.3,105.4,0.0,0.0,0.0
3,105.4,105.4,158.3,0.0,0.0,0.0
4,0.0,0.0,0.0,89.6,0.0,0.0
5,0.0,0.0,0.0,0.0,89.6,0.0
6,0.0,0.0,0.0,0.0,0.0,89.6


### Derived elastic constants (the full Materials-Project list)

In [3]:
summary = {
    'K_Voigt': d['K_Voigt'], 'K_Reuss': d['K_Reuss'], 'K_VRH': d['K_VRH'],
    'G_Voigt': d['G_Voigt'], 'G_Reuss': d['G_Reuss'], 'G_VRH': d['G_VRH'],
    "Young's E": d['youngs_modulus'], 'Poisson nu': d['poisson_ratio'],
    'Universal anisotropy A_U': d['universal_anisotropy'],
    'Mechanically stable': d['mechanically_stable'],
}
pd.Series(summary).to_frame('value (GPa where applicable)')

,value (GPa where applicable)
K_Voigt,123.024217
K_Reuss,123.024217
K_VRH,123.024217
G_Voigt,64.318988
G_Reuss,45.839841
G_VRH,55.079414
Young's E,143.780784
Poisson nu,0.305213
Universal anisotropy A_U,2.015621
Mechanically stable,True


## 2. Swapping in the GRACE-2L-SMAX foundation model
The only change is the ASE calculator. The SMAX models are cached at `/ptmp/hmai/grace_cache`
(set `GRACE_CACHE`). This cell is guarded so the notebook still runs where GRACE/TensorFlow
isn't installed; set `RUN_GRACE=1` (and ideally run on a GPU node) to execute it live.

In [4]:
import os
os.environ.setdefault('GRACE_CACHE', '/ptmp/hmai/grace_cache')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('TF_USE_LEGACY_KERAS', '1')

if os.environ.get('RUN_GRACE'):
    import sys
    sys.path.insert(0, os.path.abspath('../verification'))
    from elastic_grace_vs_mp import load_grace_calculator
    gengine = ASEEngine(EngineInput=CalcInputStatic(),
                        calculator=load_grace_calculator('GRACE-2L-SMAX-large'),
                        working_directory='elastic_grace_cu')
    gout = calculate_elastic_constants(structure=bulk('Cu','fcc',a=3.615,cubic=True),
                                       engine=gengine, relax_initial=True).run()
    gd = gout['elastic_constants']
    print('GRACE-2L-SMAX-large Cu:  K_VRH=%.1f  G_VRH=%.1f  C11=%.1f  C12=%.1f  C44=%.1f'
          % (gd['K_VRH'], gd['G_VRH'], gd['elastic_tensor_ieee'][0][0],
             gd['elastic_tensor_ieee'][0][1], gd['elastic_tensor_ieee'][3][3]))
else:
    print('Set RUN_GRACE=1 to run the GRACE-2L-SMAX-large calculator here.')

Set RUN_GRACE=1 to run the GRACE-2L-SMAX-large calculator here.


## 3. GRACE-2L-SMAX-large vs Materials Project DFT (6 elements)
The script `verification/elastic_grace_vs_mp.py` runs the workflow with GRACE-2L-SMAX-large
for Al, Cu, Si, Fe, Ni, W and cross-checks the Materials Project elastic tensors. The table
below is loaded from the committed results if present.

In [5]:
import os, pandas as pd
csv_path = os.path.join('..', 'verification', 'results', 'combined.csv')
if os.path.exists(csv_path):
    cols = ['material','mp_id','mp_state','grace_K','mp_K','exp_K',
            'grace_G','mp_G','exp_G','grace_C11','mp_C11','grace_C44','mp_C44',
            'ref_used','K_err_vs_best_%','G_err_vs_best_%','grace_stable']
    df = pd.read_csv(csv_path)
    display(df[[c for c in cols if c in df.columns]])
else:
    print('Run verification/submit_gpudev.sh + aggregate.py to populate this table.')

,material,mp_id,mp_state,grace_K,mp_K,exp_K,grace_G,mp_G,exp_G,grace_C11,mp_C11,grace_C44,mp_C44,ref_used,K_err_vs_best_%,G_err_vs_best_%,grace_stable
0,Al,mp-134,failed,76.360791,76.871,76.4,34.578535,-13.986,26.1,114.600825,70.0,39.181695,-28.0,EXP,-0.051320,32.484810,True
1,Cu,mp-30,successful,152.500725,151.394,137.1,56.294217,49.843,47.3,190.993269,186.0,87.833639,77.0,MP,0.731023,12.943076,True
2,Si,mp-149,successful,92.724993,88.916,97.8,73.440305,62.445,66.5,155.134607,153.0,99.331807,74.0,MP,4.283811,17.607984,True
3,Fe,mp-13,successful,134.894409,207.093,167.7,66.130246,67.563,82.6,184.826070,274.0,96.748992,89.0,MP,-34.862884,-2.120619,True
4,Ni,mp-23,successful,199.955879,173.808,188.0,95.398791,91.592,85.0,276.117171,249.0,134.519868,127.0,MP,15.044117,4.156249,True
5,W,mp-91,successful,334.902169,302.264,310.8,162.189272,148.152,160.2,550.177864,521.0,162.679453,138.0,MP,10.797902,9.474912,True


**Notes.** Materials Project's own elastic fit for Al (mp-134) is flagged `failed` (mechanically unstable), so the comparison uses experimental single-crystal constants there. Fe and Ni are ferromagnetic — GRACE-SMAX handles the magnetism internally. See `verification/SUMMARY.md` for the full table and interpretation.